# EDA - Online Retail II

## Pyetjet
1. Kush janë klientët më të vlefshëm dhe sa e përqendruar është shitja te ata?
2. Cilat grupe klientësh ekzistojnë sipas sjelljes së blerjes (RFM)?
3. Cilët klientë janë në rrezik humbjeje?
4. Sa do të shesë dyqani në ditët e ardhshme?
5. Cilat veçori ndikojnë më shumë në shitjet ditore?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format","{:,.2f}".format)

In [2]:
df = pd.read_csv("../data/raw/online_retail_II.csv", parse_dates=["InvoiceDate"], low_memory=False)
print(df.shape)
df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  str           
 1   StockCode    1067371 non-null  str           
 2   Description  1062989 non-null  str           
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 65.1 MB


In [4]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,"1,067,371.00",1067371,"1,067,371.00","824,364.00"
mean,9.94,2011-01-02 21:13:55.394029,4.65,"15,324.64"
min,"-80,995.00",2009-12-01 07:45:00,"-53,594.36","12,346.00"
25%,1.00,2010-07-09 09:46:00,1.25,"13,975.00"
50%,3.00,2010-12-07 15:28:00,2.10,"15,255.00"
75%,10.00,2011-07-22 10:23:00,4.15,"16,797.00"
max,"80,995.00",2011-12-09 12:50:00,"38,970.00","18,287.00"
std,172.71,NaN,123.55,"1,697.46"


In [5]:
pd.DataFrame({
    "missing": df.isna().sum(),
    "pct": (df.isna().mean()*100).round(2)
})

,missing,pct
Invoice,0,0.00
StockCode,0,0.00
Description,4382,0.41
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,243007,22.77
Country,0,0.00


In [8]:
print("Fatura unike: ", df["Invoice"].nunique())
print("Klient unik: ", df["Customer ID"].nunique())
print("Produkte unike: ", df["StockCode"].nunique())
print("Vende: ", df["Country"].nunique())
print("Periudha: ", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())

Fatura unike:  53628
Klient unik:  5942
Produkte unike:  5305
Vende:  43
Periudha:  2009-12-01 07:45:00 -> 2011-12-09 12:50:00


In [9]:
print(df["Country"].value_counts().head(10))
print("Rreshta kthimesh (fatura me C): ", df["Invoice"].astype(str).str.startswith("C").sum())

Country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Spain               3811
Switzerland         3189
Belgium             3123
Portugal            2620
Australia           1913
Name: count, dtype: int64
Rreshta kthimesh (fatura me C):  19494


## Vëzhgimet e Ditës 1

- Dataseti ka **1,067,371 rreshta** dhe **8 kolona**, i bashkuar nga dy fletë Excel (2009–2010: 525,461 rreshta; 2010–2011: 541,910 rreshta).
- Të dhënat mbulojnë periudhën nga dhjetori 2009 deri në dhjetor 2011 (datat e sakta: 2009-12-01 deri më 2011-12-09).
- Një faturë përmban disa rreshta, sepse çdo rresht është një produkt brenda një fature. Ka 53628 fatura unike, 5942 klientë unikë dhe 5305 produkte unike.
- **Customer ID mungon në 22.77% të rreshtave.** Këta rreshta nuk mund të përdoren në analizën e klientëve, por mbeten të vlefshëm për shitjet totale.
- `Quantity` dhe `Price` kanë vlera negative (minimumi i `Quantity` është -80995, ai i `Price` është -53,594.36). Të paktën një pjesë e tyre lidhen me fatura anuluese (numri i faturës fillon me "C"), dhe ka 19494 rreshta të tillë.
- Vendi me më shumë rreshta është **United Kingdom**, me 91.9% të të gjithë rreshtave.